# MAI-Transcribe-2: speaker-aware transcription

## 1. Reuse the MAI-Transcribe-1.5 setup

How can we move an existing MAI-Transcribe-1.5 workflow to MAI-Transcribe-2 and inspect the features that distinguish it? This notebook keeps the 1.5 setup and examples, then adds speaker diarization, word timestamps, and a controlled model comparison.

MAI-Transcribe runs through the **LLM Speech API**, currently in public preview with no SLA. No model deployment is required; the request selects the model.

**Before we begin**

- Pricing: $0.10 per audio hour for a limited time; standard pricing is $0.36 per hour. Confirm current rates on the [model page](https://microsoft.ai/models/mai-transcribe-2/).
- Supported input formats: WAV, MP3, and FLAC, up to 300 MB.
- We need a Microsoft Foundry AI Services/Speech resource in a supported region. See the [quickstart](../../quickstart/README.md).
- Set `MICROSOFT_FOUNDRY_ENDPOINT` and optionally `MICROSOFT_FOUNDRY_API_KEY`. Use `AZURE_SPEECH_ENDPOINT` when Speech uses a separate resource.

The examples reuse the audio fixtures from the MAI-Transcribe-1.5 capsule.

In [1]:
%pip install azure-ai-transcription azure-identity python-dotenv pandas --quiet

import json
import os
import time
from collections.abc import Mapping
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from azure.ai.transcription import TranscriptionClient
from azure.ai.transcription.models import (
    EnhancedModeProperties,
    PhraseListProperties,
    TranscriptionContent,
    TranscriptionOptions,
)
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

foundry_endpoint = os.environ.get("MICROSOFT_FOUNDRY_ENDPOINT")
speech_endpoint = os.environ.get("AZURE_SPEECH_ENDPOINT")
if not (speech_endpoint or foundry_endpoint):
    raise EnvironmentError(
        "Set MICROSOFT_FOUNDRY_ENDPOINT or AZURE_SPEECH_ENDPOINT before continuing."
    )
if not speech_endpoint:
    parsed = urlparse(foundry_endpoint)
    speech_endpoint = f"{parsed.scheme}://{parsed.netloc}"

api_key = os.environ.get("AZURE_SPEECH_API_KEY") or os.environ.get(
    "MICROSOFT_FOUNDRY_API_KEY"
)
if api_key:
    credential = AzureKeyCredential(api_key)
    auth_method = "API key"
else:
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    auth_method = "Entra ID (DefaultAzureCredential)"

MODEL_15 = "mai-transcribe-1.5"
MODEL_2 = "MAI-Transcribe-2"

cwd = Path.cwd()
NOTEBOOK_DIR = cwd if cwd.name == "mai-transcribe-2" else cwd / "models/microsoft-ai/mai-transcribe-2"
DATA_DIR = NOTEBOOK_DIR.parent / "mai-transcribe-1.5" / "data"
OUT_DIR = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(exist_ok=True)

client = TranscriptionClient(endpoint=speech_endpoint, credential=credential)
print(f"Endpoint: {speech_endpoint}")
print(f"Auth: {auth_method}")
print(f"Audio: {DATA_DIR.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")

Note: you may need to restart the kernel to use updated packages.
Endpoint: https://model-releases-proj-resource.services.ai.azure.com
Auth: Entra ID (DefaultAzureCredential)
Audio: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-1.5/data
Output: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-2/output


## 2. Configure MAI-Transcribe-1.5 and 2

Both versions use the same endpoint, credential, and SDK client. `make_options` keeps model selection in `enhanced_mode` and adds MAI-Transcribe-2 preview fields by their wire names.

For the version comparison, we deliberately omit model-exclusive features. This keeps the input audio, locale, and request shape equivalent so differences reflect the selected model rather than diarization or timestamp processing.

In [2]:
def value(item, *names, default=None):
    """Read a field from an SDK mapping or model object."""
    for name in names:
        if isinstance(item, Mapping) and name in item:
            return item[name]
        candidate = getattr(item, name, None)
        if candidate is not None:
            return candidate
    return default


def make_options(
    model_name,
    *,
    locales=None,
    phrases=None,
    biasing_weight=None,
    diarization=False,
    timestamps=None,
    transcribe_style=None,
):
    enhanced = EnhancedModeProperties()
    enhanced["model"] = model_name

    settings = {"enhanced_mode": enhanced}
    if locales:
        settings["locales"] = locales
    if phrases:
        settings["phrase_list"] = PhraseListProperties(
            phrases=phrases,
            biasing_weight=biasing_weight if biasing_weight is not None else 1.0,
        )

    options = TranscriptionOptions(**settings)
    if diarization:
        options["diarization"] = {"enabled": True}
    model_options = {}
    if timestamps:
        model_options["timestamps"] = timestamps
    if transcribe_style:
        model_options["transcribeStyle"] = transcribe_style
    if model_options:
        options["modelOptions"] = model_options
    return options


def transcribe(audio_path, *, model_name=MODEL_2, **kwargs):
    with Path(audio_path).open("rb") as audio:
        content = TranscriptionContent(
            definition=make_options(model_name, **kwargs),
            audio=audio,
        )
        return client.transcribe(content)


def transcript_text(result):
    combined = value(result, "combined_phrases", "combinedPhrases", default=[])
    return value(combined[0], "text", default="") if combined else ""


def phrase_rows(result):
    rows = []
    for phrase in value(result, "phrases", default=[]) or []:
        offset = value(phrase, "offset_milliseconds", "offsetMilliseconds", default=0)
        duration = value(phrase, "duration_milliseconds", "durationMilliseconds", default=0)
        rows.append(
            {
                "speaker": value(phrase, "speaker", "speaker_id", "speakerId"),
                "start_seconds": round(offset / 1000, 3),
                "end_seconds": round((offset + duration) / 1000, 3),
                "locale": value(phrase, "locale"),
                "confidence": value(phrase, "confidence"),
                "text": value(phrase, "text", default=""),
            }
        )
    return rows


def word_rows(result):
    rows = []
    for phrase in value(result, "phrases", default=[]) or []:
        phrase_speaker = value(phrase, "speaker", "speaker_id", "speakerId")
        for word in value(phrase, "words", default=[]) or []:
            offset = value(word, "offset_milliseconds", "offsetMilliseconds", default=0)
            duration = value(word, "duration_milliseconds", "durationMilliseconds", default=0)
            rows.append(
                {
                    "speaker": value(word, "speaker", "speaker_id", "speakerId", default=phrase_speaker),
                    "word": value(word, "word", "text", default=""),
                    "start_seconds": round(offset / 1000, 3),
                    "end_seconds": round((offset + duration) / 1000, 3),
                }
            )
    return rows


def timed_transcription(audio_path, *, model_name=MODEL_2, **kwargs):
    started = time.perf_counter()
    result = transcribe(audio_path, model_name=model_name, **kwargs)
    elapsed = time.perf_counter() - started
    duration_ms = value(result, "duration_milliseconds", "durationMilliseconds", default=0)
    duration_seconds = duration_ms / 1000
    return result, elapsed, duration_seconds

## 3. Load and validate audio

We validate the input before spending a service call. The cell checks the path, extension, size, duration, sample rate, and channel count. Change `sample_audio` to another WAV, MP3, or FLAC fixture to test a different recording.

In [3]:
%pip install soundfile --quiet

import soundfile as sf

SUPPORTED_FORMATS = {".wav", ".mp3", ".flac"}
sample_audio = DATA_DIR / "normal - on call agent building.mp3"

if not sample_audio.is_file():
    raise FileNotFoundError(f"Audio fixture not found: {sample_audio.resolve()}")
if sample_audio.suffix.lower() not in SUPPORTED_FORMATS:
    raise ValueError(f"Expected one of {sorted(SUPPORTED_FORMATS)}, got {sample_audio.suffix}")
if sample_audio.stat().st_size > 300 * 1024 * 1024:
    raise ValueError("Audio exceeds the 300 MB request limit")

audio_info = sf.info(sample_audio)
audio_metadata = {
    "path": str(sample_audio.resolve()),
    "format": audio_info.format,
    "duration_seconds": round(audio_info.duration, 2),
    "sample_rate_hz": audio_info.samplerate,
    "channels": audio_info.channels,
    "size_mb": round(sample_audio.stat().st_size / (1024 * 1024), 2),
}
display(pd.DataFrame([audio_metadata]))

Note: you may need to restart the kernel to use updated packages.


,path,format,duration_seconds,sample_rate_hz,channels,size_mb
0,/workspaces/model-releases-public/models/micro...,MP3,49.82,48000,1,1.52


## 4. Call MAI-Transcribe-2

This first call requests both speaker diarization and word timestamps. Diarization is best suited to shorter recordings during preview; long recordings can time out. The response is kept intact for later export.

The next examples retain the useful 1.5 workflow: readable versus verbatim text, phrase-list biasing, and multilingual recognition.

In [4]:
mai2_result, mai2_latency, mai2_audio_seconds = timed_transcription(
    sample_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    diarization=True,
    timestamps="word",
)

print(transcript_text(mai2_result))
print(f"Audio: {mai2_audio_seconds:.2f}s | Latency: {mai2_latency:.2f}s")

Hello, I am building an on-call agent for DevOps and the idea is to get engineers paid for when, okay, hear me out. When engineers are paid for incidents, in the technically spend like 10-15 minutes understanding the context, checking dashboards, logs, deploys and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and yeah, does a blind execution of something?
Audio: 49.82s | Latency: 6.23s


## 5. Extract speaker diarization and timestamps

Diarization answers **who spoke when**. Word timestamps answer **when each recognized token occurred**. We normalize both SDK object and mapping responses, sort by start time, and keep seconds numeric so downstream code can filter or seek accurately.

Speaker labels identify distinct voices within this recording; they do not identify real people.

In [5]:
diarized_columns = [
    "speaker",
    "start_seconds",
    "end_seconds",
    "locale",
    "confidence",
    "text",
]
word_columns = ["speaker", "word", "start_seconds", "end_seconds"]
diarized_df = pd.DataFrame(phrase_rows(mai2_result), columns=diarized_columns)
words_df = pd.DataFrame(word_rows(mai2_result), columns=word_columns)
if not diarized_df.empty:
    diarized_df = diarized_df.sort_values("start_seconds", ignore_index=True)
if not words_df.empty:
    words_df = words_df.sort_values("start_seconds", ignore_index=True)

print("Diarized segments")
display(diarized_df)
print("Word timestamps")
display(words_df.head(30))

Diarized segments


,speaker,start_seconds,end_seconds,locale,confidence,text
0,1,8.11,20.91,en-US,0.802941,"Hello, I am building an on-call agent for DevO..."
1,1,21.35,24.03,en-US,0.802941,"When engineers are paid for incidents,"
2,1,24.75,36.59,en-US,0.794269,in the technically spend like 10-15 minutes un...
3,1,37.03,48.59,en-US,0.794269,What if you had an agent that automates that t...


Word timestamps


,speaker,word,start_seconds,end_seconds
0,1,"Hello,",8.11,8.67
1,1,I,9.31,9.35
2,1,am,9.35,9.79
3,1,building,9.79,10.67
4,1,an,11.03,11.15
5,1,on-call,11.15,11.95
6,1,agent,12.31,12.83
7,1,for,12.83,12.99
8,1,DevOps,12.99,13.63
9,1,and,14.03,14.35


### Retain the MAI-Transcribe-1.5 workflow

MAI-Transcribe-2 retains the practical controls demonstrated in the 1.5 notebook:

- Compare readability-oriented and verbatim transcript styles on disfluent speech.
- Bias specialized vocabulary with a short `phraseList`.
- Let the model identify language automatically or pin a full BCP-47 locale such as `es-ES`.

These are separate calls so each option’s effect remains visible.

In [6]:
style_audio = DATA_DIR / "filler words for verbatim.mp3"
readable_result = transcribe(
    style_audio,
    model_name=MODEL_2,
    locales=["en-US"],
)
verbatim_result = transcribe(
    style_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    transcribe_style="verbatim",
)

phrases = ["Microsoft Foundry", "AI", "Speech Recognition", "DevOps", "Copilot"]
bias_audio = DATA_DIR / "rain + specific content.mp3"
unbiased_result = transcribe(bias_audio, model_name=MODEL_2, locales=["en-US"])
biased_result = transcribe(
    bias_audio,
    model_name=MODEL_2,
    locales=["en-US"],
    phrases=phrases,
    biasing_weight=1.5,
)

spanish_audio = DATA_DIR / "Multilingual Speech Transcript - Spanish.mp3"
auto_language_result = transcribe(spanish_audio, model_name=MODEL_2)
pinned_language_result = transcribe(
    spanish_audio,
    model_name=MODEL_2,
    locales=["es-ES"],
)

feature_examples = pd.DataFrame(
    [
        {"example": "readability default", "transcript": transcript_text(readable_result)},
        {"example": "verbatim", "transcript": transcript_text(verbatim_result)},
        {"example": "without phrase list", "transcript": transcript_text(unbiased_result)},
        {"example": "with phrase list", "transcript": transcript_text(biased_result)},
        {"example": "Spanish auto-detect", "transcript": transcript_text(auto_language_result)},
        {"example": "Spanish pinned", "transcript": transcript_text(pinned_language_result)},
    ]
)
display(feature_examples)

,example,transcript
0,readability default,"Hello, I am Bethany and I wanted to share an i..."
1,verbatim,"Hello, I am Bethany and I wanted to share an i..."
2,without phrase list,"Hello, my name is Bethany and I'm recording th..."
3,with phrase list,"Hello, my name is Bethany and I'm recording th..."
4,Spanish auto-detect,"Hola, me llamo Gustavo y estoy grabando este m..."
5,Spanish pinned,"Hola, me llamo Gustavo y estoy grabando este m..."


## 6. Compare MAI-Transcribe-1.5 with 2

We send the same file with the same pinned locale to both models. No 2-only option is enabled in these two calls. The comparison captures latency, real-time factor (RTF), transcript text, phrase timing coverage, speaker labels, detected locales, confidence, and errors.

This is a workload check, not a benchmark claim. Run several representative files and repeat calls before drawing conclusions about quality or speed.

In [7]:
comparison_records = []
comparison_results = {}

for model_name in (MODEL_15, MODEL_2):
    started = time.perf_counter()
    try:
        result, latency, audio_seconds = timed_transcription(
            sample_audio,
            model_name=model_name,
            locales=["en-US"],
        )
        comparison_results[model_name] = result
        phrases_for_model = phrase_rows(result)
        words_for_model = word_rows(result)
        confidences = [
            row["confidence"]
            for row in phrases_for_model
            if row["confidence"] is not None
        ]
        comparison_records.append(
            {
                "model": model_name,
                "audio_seconds": round(audio_seconds, 2),
                "latency_seconds": round(latency, 2),
                "rtf": round(latency / audio_seconds, 3) if audio_seconds else None,
                "detected_locales": ", ".join(
                    sorted({row["locale"] for row in phrases_for_model if row["locale"]})
                ),
                "mean_confidence": round(sum(confidences) / len(confidences), 3)
                if confidences
                else None,
                "timestamped_phrases": sum(
                    row["end_seconds"] > row["start_seconds"] for row in phrases_for_model
                ),
                "timestamped_words": len(words_for_model),
                "speakers": ", ".join(
                    sorted({str(row["speaker"]) for row in phrases_for_model if row["speaker"] is not None})
                ),
                "transcript": transcript_text(result),
                "error": None,
            }
        )
    except Exception as exc:
        comparison_records.append(
            {
                "model": model_name,
                "audio_seconds": None,
                "latency_seconds": round(time.perf_counter() - started, 2),
                "rtf": None,
                "detected_locales": "",
                "mean_confidence": None,
                "timestamped_phrases": 0,
                "timestamped_words": 0,
                "speakers": "",
                "transcript": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

comparison_df = pd.DataFrame(comparison_records)
display(comparison_df.drop(columns="transcript"))
for row in comparison_records:
    print(f"\n--- {row['model']} ---")
    print(row["error"] or row["transcript"])

comparison_df.to_csv(OUT_DIR / "mai-transcribe-1.5-vs-2.csv", index=False)

,model,audio_seconds,latency_seconds,rtf,detected_locales,mean_confidence,timestamped_phrases,timestamped_words,speakers,error
0,mai-transcribe-1.5,49.82,2.79,0.056,en-US,0.798,4,70,,None
1,MAI-Transcribe-2,49.82,2.53,0.051,en-US,0.798,4,70,,None



--- mai-transcribe-1.5 ---
Hello, I am building an on-call agent for DevOps and the idea is to get engineers paid for when, okay, hear me out. When engineers are paid for incidents, in the technically spend like 10-15 minutes understanding the context, checking dashboards, logs, deploys and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and yeah, does a blind execution of something?

--- MAI-Transcribe-2 ---
Hello, I am building an on-call agent for DevOps and the idea is to get engineers paid for when, okay, hear me out. When engineers are paid for incidents, in the technically spend like 10-15 minutes understanding the context, checking dashboards, logs, deploys and related alerts before they can start diagnosing. What if you had an agent that automates that triage process and yeah, does a blind execution of something?


## 7. Display and export results

We export machine-readable artifacts for downstream review:

- The raw MAI-Transcribe-2 response as JSON
- Timestamped words as CSV
- Diarized segments as CSV
- Side-by-side comparison metrics and transcripts as CSV

The raw response preserves preview fields that our normalized tables may not yet expose.

In [8]:
def jsonable(item):
    if hasattr(item, "as_dict"):
        return jsonable(item.as_dict())
    if isinstance(item, Mapping):
        return {str(key): jsonable(val) for key, val in item.items()}
    if isinstance(item, (list, tuple)):
        return [jsonable(entry) for entry in item]
    return item


raw_path = OUT_DIR / "mai-transcribe-2-response.json"
raw_path.write_text(
    json.dumps(jsonable(mai2_result), indent=2, default=str),
    encoding="utf-8",
)
diarized_path = OUT_DIR / "mai-transcribe-2-diarized.csv"
words_path = OUT_DIR / "mai-transcribe-2-word-timestamps.csv"
comparison_path = OUT_DIR / "mai-transcribe-1.5-vs-2.csv"

diarized_df.to_csv(diarized_path, index=False)
words_df.to_csv(words_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

for path in (raw_path, diarized_path, words_path, comparison_path):
    print(f"Saved: {path}")

Saved: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-2/output/mai-transcribe-2-response.json
Saved: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-2/output/mai-transcribe-2-diarized.csv
Saved: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-2/output/mai-transcribe-2-word-timestamps.csv
Saved: /workspaces/model-releases-public/models/microsoft-ai/mai-transcribe-2/output/mai-transcribe-1.5-vs-2.csv


### Your Turn to Explore

1. Replace `sample_audio` with a short meeting recording and compare the returned speaker count with the voices we hear.
2. Generate captions from `words_df`, grouping words into readable time windows.
3. Run the comparison over several noisy and multilingual clips, then summarize median RTF and review transcript differences against hand-verified references.

### Summary

We reused the MAI-Transcribe-1.5 client pattern with MAI-Transcribe-2, validated audio before submission, requested speaker diarization and word timestamps, normalized both into tables, and retained transcript style, phrase-list, and multilingual examples. We also compared 1.5 and 2 under equivalent baseline settings and exported the results for review.

Reach for MAI-Transcribe-2 when speaker turns or precise alignment matter alongside multilingual recognition and domain vocabulary. Because the service is in preview, validate feature limits, response shape, accuracy, latency, and current pricing against our own workload before production use.

See the [Audio / Speech primer](../../../docs/primers/audio-speech.md) and [glossary](../../../docs/GLOSSARY.md) for related concepts.

### References

- [MAI-Transcribe-2 model page](https://microsoft.ai/models/mai-transcribe-2/) - features, language coverage, benchmarks, and current pricing.
- [MAI-Transcribe-2 announcement](https://microsoft.ai/news/mai-transcribe-2-is-the-fastest-most-accurate-and-cheapest-speech-recognition-model-in-the-world/) - release context and comparative results.
- [MAI-Transcribe in Azure Speech](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry) - request parameters, diarization, timestamps, styles, and limits.
- [LLM Speech transcription and translation](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/llm-speech?tabs=new-foundry%2Cwindows&pivots=programming-language-python) - authentication, Python SDK, and response handling.
- [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech) - current endpoint and regional availability guidance.